# Rule-based Risk Scoring

**Người thực hiện:** Yến

## Mục tiêu

Xây dựng hệ thống chấm điểm nguy cơ bỏ học dựa trên các yếu tố học tập, chuyên cần, tâm lý và tài chính. Kết quả gồm điểm rủi ro, mức cảnh báo, nguyên nhân và khuyến nghị.

### 1. Đọc và kiểm tra dữ liệu

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/student_dropout.csv")

In [3]:
print("Kích thước dữ liệu:", df.shape)
df.head()

Kích thước dữ liệu: (10000, 19)


,Student_ID,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,1,22.1,Male,25000.0,Yes,3.36,86.1,2,20.4,Yes,No,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,2,20.7,Male,25000.0,Yes,4.30,68.0,2,44.0,No,No,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,3,22.4,Male,40183.0,Yes,4.40,70.9,0,48.9,Yes,No,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0
3,4,24.4,Male,NaN,Yes,NaN,82.2,2,38.6,No,No,NaN,1.78,1.77,1.77,Year 1,CS,High School,1
4,5,20.5,Female,25319.0,Yes,4.19,75.7,1,23.0,No,No,7.0,1.48,0.91,0.87,Year 4,Business,Bachelor,0


In [4]:
df.columns.tolist()

['Student_ID',
 'Age',
 'Gender',
 'Family_Income',
 'Internet_Access',
 'Study_Hours_per_Day',
 'Attendance_Rate',
 'Assignment_Delay_Days',
 'Travel_Time_Minutes',
 'Part_Time_Job',
 'Scholarship',
 'Stress_Index',
 'GPA',
 'Semester_GPA',
 'CGPA',
 'Semester',
 'Department',
 'Parental_Education',
 'Dropout']

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Student_ID             10000 non-null  int64  
 1   Age                    10000 non-null  float64
 2   Gender                 10000 non-null  str    
 3   Family_Income          9500 non-null   float64
 4   Internet_Access        10000 non-null  str    
 5   Study_Hours_per_Day    9500 non-null   float64
 6   Attendance_Rate        10000 non-null  float64
 7   Assignment_Delay_Days  10000 non-null  int64  
 8   Travel_Time_Minutes    10000 non-null  float64
 9   Part_Time_Job          10000 non-null  str    
 10  Scholarship            10000 non-null  str    
 11  Stress_Index           9500 non-null   float64
 12  GPA                    10000 non-null  float64
 13  Semester_GPA           10000 non-null  float64
 14  CGPA                   10000 non-null  float64
 15  Semester      

### 2. Kiểm tra missing values
Mục đích: Xác định những thuộc tính bị thiếu dữ liệu trước khi xây dựng luật chấm điểm. Nếu không xử lý, việc so sánh điều kiện có thể cho kết quả không chính xác.

In [6]:
missing_values = df.isnull().sum()
missing_values[missing_values > 0]

Family_Income          500
Study_Hours_per_Day    500
Stress_Index           500
Parental_Education     511
dtype: int64

### 3. Phân tích biến mục tiêu Dropout

In [7]:
dropout_count = df["Dropout"].value_counts()
dropout_rate = df["Dropout"].value_counts(normalize=True) * 100

print("Số lượng sinh viên theo trạng thái:")
print(dropout_count.rename(index={
    0: "Không bỏ học",
    1: "Bỏ học"
}))

print("\nTỷ lệ sinh viên theo trạng thái (%):")
print(dropout_rate.rename(index={
    0: "Không bỏ học",
    1: "Bỏ học"
}).round(2))

Số lượng sinh viên theo trạng thái:
Dropout
Không bỏ học    7646
Bỏ học          2354
Name: count, dtype: int64

Tỷ lệ sinh viên theo trạng thái (%):
Dropout
Không bỏ học    76.46
Bỏ học          23.54
Name: proportion, dtype: float64


### 4. Xử lý dữ liệu thiếu
Các cột dạng số được điền bằng giá trị trung vị vì trung vị ít bị ảnh hưởng bởi các giá trị quá lớn hoặc quá nhỏ. Cột dạng phân loại được điền bằng `Unknown` để giữ lại sinh viên nhưng vẫn thể hiện rằng thông tin chưa được cung cấp.

In [8]:
# Tạo bản sao để không làm thay đổi dữ liệu gốc
df_clean = df.copy()

numeric_missing_columns = [
    "Family_Income",
    "Study_Hours_per_Day",
    "Stress_Index"
]

for column in numeric_missing_columns:
    median_value = df_clean[column].median()
    df_clean[column] = df_clean[column].fillna(median_value)

df_clean["Parental_Education"] = (
    df_clean["Parental_Education"].fillna("Unknown")
)

In [9]:
remaining_missing = df_clean.isnull().sum().sum()

print("Tổng số giá trị còn thiếu:", remaining_missing)

Tổng số giá trị còn thiếu: 0


### 5. Phân tích các yếu tố rủi ro dạng số

Mục đích: So sánh giá trị trung bình của các yếu tố học tập, chuyên cần, tâm lý và tài chính giữa nhóm bỏ học và không bỏ học. Kết quả này được dùng làm cơ sở lựa chọn biến và xây dựng luật chấm điểm rủi ro.

In [10]:
risk_features = [
    "GPA",
    "Semester_GPA",
    "CGPA",
    "Attendance_Rate",
    "Study_Hours_per_Day",
    "Assignment_Delay_Days",
    "Stress_Index",
    "Family_Income"
]

risk_comparison = (
    df_clean
    .groupby("Dropout")[risk_features]
    .mean()
    .T
    .rename(columns={
        0: "Không bỏ học",
        1: "Bỏ học"
    })
)

risk_comparison["Chênh lệch"] = (
    risk_comparison["Bỏ học"]
    - risk_comparison["Không bỏ học"]
)

risk_comparison.round(2)

Dropout,Không bỏ học,Bỏ học,Chênh lệch
GPA,2.58,1.43,-1.15
Semester_GPA,2.57,1.44,-1.13
CGPA,2.56,1.44,-1.12
Attendance_Rate,82.48,79.31,-3.17
Study_Hours_per_Day,4.07,3.82,-0.26
Assignment_Delay_Days,1.74,2.00,0.26
Stress_Index,5.27,6.28,1.01
Family_Income,38060.35,37572.07,-488.28


#### Nhận xét

Nhóm sinh viên bỏ học có GPA, CGPA, tỷ lệ chuyên cần và thời gian tự học trung bình thấp hơn nhóm không bỏ học. Ngược lại, nhóm bỏ học có mức độ căng thẳng và số ngày nộp bài trễ cao hơn. Trong đó, GPA và Stress Index cho thấy sự khác biệt rõ ràng hơn và nên được ưu tiên khi xây dựng luật chấm điểm.

### 6. Phân tích các yếu tố rủi ro dạng phân loại

Mục đích: So sánh tỷ lệ bỏ học theo tình trạng làm thêm, học bổng và khả năng truy cập Internet. Kết quả giúp xác định các yếu tố này có nên được đưa vào luật chấm điểm hay không.

In [11]:
from IPython.display import display

categorical_features = [
    "Part_Time_Job",
    "Scholarship",
    "Internet_Access"
]

for feature in categorical_features:
    dropout_by_category = pd.crosstab(
        df_clean[feature],
        df_clean["Dropout"],
        normalize="index"
    ) * 100

    dropout_by_category = dropout_by_category.rename(columns={
        0: "Không bỏ học (%)",
        1: "Bỏ học (%)"
    })

    print(f"\nYếu tố: {feature}")
    display(dropout_by_category.round(2))


Yếu tố: Part_Time_Job


Dropout,Không bỏ học (%),Bỏ học (%)
Part_Time_Job,,
No,77.74,22.26
Yes,74.55,25.45



Yếu tố: Scholarship


Dropout,Không bỏ học (%),Bỏ học (%)
Scholarship,,
No,76.25,23.75
Yes,76.84,23.16



Yếu tố: Internet_Access


Dropout,Không bỏ học (%),Bỏ học (%)
Internet_Access,,
No,71.57,28.43
Yes,77.15,22.85


#### Nhận xét

Sinh viên không có Internet có tỷ lệ bỏ học cao hơn sinh viên có Internet. Sinh viên làm thêm cũng có tỷ lệ bỏ học cao hơn nhưng mức chênh lệch tương đối nhỏ.

Tỷ lệ bỏ học giữa nhóm có và không có học bổng gần như tương đương, vì vậy Scholarship không được đưa vào phiên bản đầu tiên của bộ luật. Internet Access có thể được sử dụng như một yếu tố rủi ro, còn Part-time Job chỉ nên có trọng số thấp.

### 7. Phân tích ngưỡng của các yếu tố rủi ro

Mục đích: Chia các yếu tố thành những khoảng dễ giải thích và tính tỷ lệ bỏ học trong từng khoảng. Kết quả được sử dụng để lựa chọn điều kiện và trọng số cho bộ luật.

In [12]:
def analyze_risk_bands(feature, bins, labels):
    risk_band = pd.cut(
        df_clean[feature],
        bins=bins,
        labels=labels,
        right=False,
        include_lowest=True
    )

    result = (
        df_clean
        .assign(Risk_Band=risk_band)
        .groupby("Risk_Band", observed=True)["Dropout"]
        .agg(
            Student_Count="count",
            Dropout_Count="sum",
            Dropout_Rate="mean"
        )
    )

    result["Dropout_Rate"] = (
        result["Dropout_Rate"] * 100
    ).round(2)

    print(f"\nYếu tố: {feature}")
    display(result)

In [13]:
analyze_risk_bands(
    "GPA",
    bins=[float("-inf"), 2.0, 2.5, float("inf")],
    labels=["< 2.0", "2.0 - < 2.5", ">= 2.5"]
)

analyze_risk_bands(
    "Attendance_Rate",
    bins=[float("-inf"), 75, 85, float("inf")],
    labels=["< 75", "75 - < 85", ">= 85"]
)

analyze_risk_bands(
    "Stress_Index",
    bins=[float("-inf"), 5, 7, float("inf")],
    labels=["< 5", "5 - < 7", ">= 7"]
)

analyze_risk_bands(
    "Study_Hours_per_Day",
    bins=[float("-inf"), 2, 4, float("inf")],
    labels=["< 2", "2 - < 4", ">= 4"]
)

analyze_risk_bands(
    "Assignment_Delay_Days",
    bins=[float("-inf"), 1, 3, float("inf")],
    labels=["0 ngày", "1 - 2 ngày", ">= 3 ngày"]
)


Yếu tố: GPA


,Student_Count,Dropout_Count,Dropout_Rate
Risk_Band,,,
< 2.0,3863,1737,44.97
2.0 - < 2.5,1641,304,18.53
>= 2.5,4496,313,6.96



Yếu tố: Attendance_Rate


,Student_Count,Dropout_Count,Dropout_Rate
Risk_Band,,,
< 75,2004,690,34.43
75 - < 85,4510,1104,24.48
>= 85,3486,560,16.06



Yếu tố: Stress_Index


,Student_Count,Dropout_Count,Dropout_Rate
Risk_Band,,,
< 5,3582,456,12.73
5 - < 7,4427,1106,24.98
>= 7,1991,792,39.78



Yếu tố: Study_Hours_per_Day


,Student_Count,Dropout_Count,Dropout_Rate
Risk_Band,,,
< 2,582,187,32.13
2 - < 4,4153,1088,26.20
>= 4,5265,1079,20.49



Yếu tố: Assignment_Delay_Days


,Student_Count,Dropout_Count,Dropout_Rate
Risk_Band,,,
0 ngày,1683,324,19.25
1 - 2 ngày,5614,1263,22.50
>= 3 ngày,2703,767,28.38


In [14]:
analyze_risk_bands(
    "Family_Income",
    bins=[float("-inf"), 30000, 50000, float("inf")],
    labels=["< 30000", "30000 - < 50000", ">= 50000"]
)


Yếu tố: Family_Income


,Student_Count,Dropout_Count,Dropout_Rate
Risk_Band,,,
< 30000,5309,1268,23.88
30000 - < 50000,2860,681,23.81
>= 50000,1831,405,22.12


#### Kết luận lựa chọn yếu tố

Các yếu tố được chọn cho bộ luật gồm GPA, Attendance Rate, Stress Index, Study Hours, Assignment Delay, Internet Access và Part-time Job.

Family Income và Scholarship không được chọn vì tỷ lệ bỏ học giữa các nhóm chỉ chênh lệch nhỏ. Semester GPA và CGPA không được cộng điểm riêng để tránh tính lặp với GPA.

### 8. Thiết kế bảng luật và trọng số

Điểm số được gán dựa trên mức chênh lệch tỷ lệ bỏ học:

- `3 điểm`: Yếu tố rủi ro rất mạnh.
- `2 điểm`: Yếu tố rủi ro mạnh.
- `1 điểm`: Yếu tố bổ sung hoặc cần theo dõi.

Các điều kiện thuộc cùng một yếu tố loại trừ lẫn nhau. Ví dụ, một sinh viên chỉ nhận một mức điểm GPA.

In [15]:
rules_table = pd.DataFrame([
    {
        "Yếu tố": "GPA",
        "Điều kiện": "GPA < 2.0",
        "Điểm": 3,
        "Mức ảnh hưởng": "Rất mạnh"
    },
    {
        "Yếu tố": "GPA",
        "Điều kiện": "2.0 <= GPA < 2.5",
        "Điểm": 1,
        "Mức ảnh hưởng": "Theo dõi"
    },
    {
        "Yếu tố": "Attendance_Rate",
        "Điều kiện": "Attendance_Rate < 75",
        "Điểm": 2,
        "Mức ảnh hưởng": "Mạnh"
    },
    {
        "Yếu tố": "Attendance_Rate",
        "Điều kiện": "75 <= Attendance_Rate < 85",
        "Điểm": 1,
        "Mức ảnh hưởng": "Theo dõi"
    },
    {
        "Yếu tố": "Stress_Index",
        "Điều kiện": "Stress_Index >= 7",
        "Điểm": 2,
        "Mức ảnh hưởng": "Mạnh"
    },
    {
        "Yếu tố": "Stress_Index",
        "Điều kiện": "5 <= Stress_Index < 7",
        "Điểm": 1,
        "Mức ảnh hưởng": "Theo dõi"
    },
    {
        "Yếu tố": "Study_Hours_per_Day",
        "Điều kiện": "Study_Hours_per_Day < 2",
        "Điểm": 1,
        "Mức ảnh hưởng": "Bổ sung"
    },
    {
        "Yếu tố": "Assignment_Delay_Days",
        "Điều kiện": "Assignment_Delay_Days >= 3",
        "Điểm": 1,
        "Mức ảnh hưởng": "Bổ sung"
    },
    {
        "Yếu tố": "Internet_Access",
        "Điều kiện": "Internet_Access == 'No'",
        "Điểm": 1,
        "Mức ảnh hưởng": "Bổ sung"
    },
    {
        "Yếu tố": "Part_Time_Job",
        "Điều kiện": "Part_Time_Job == 'Yes'",
        "Điểm": 1,
        "Mức ảnh hưởng": "Bổ sung"
    }
])

rules_table

,Yếu tố,Điều kiện,Điểm,Mức ảnh hưởng
0,GPA,GPA < 2.0,3,Rất mạnh
1,GPA,2.0 <= GPA < 2.5,1,Theo dõi
2,Attendance_Rate,Attendance_Rate < 75,2,Mạnh
3,Attendance_Rate,75 <= Attendance_Rate < 85,1,Theo dõi
4,Stress_Index,Stress_Index >= 7,2,Mạnh
5,Stress_Index,5 <= Stress_Index < 7,1,Theo dõi
6,Study_Hours_per_Day,Study_Hours_per_Day < 2,1,Bổ sung
7,Assignment_Delay_Days,Assignment_Delay_Days >= 3,1,Bổ sung
8,Internet_Access,Internet_Access == 'No',1,Bổ sung
9,Part_Time_Job,Part_Time_Job == 'Yes',1,Bổ sung


### 9. Xây dựng hàm Rule-based Scoring

Mỗi sinh viên được kiểm tra lần lượt qua các luật. Khi thỏa mãn một điều kiện, hệ thống cộng điểm, lưu nguyên nhân và tạo khuyến nghị tương ứng.

Mức cảnh báo ban đầu:

- `0 - 2 điểm`: Nguy cơ thấp.
- `3 - 5 điểm`: Nguy cơ trung bình.
- `6 điểm trở lên`: Nguy cơ cao.

Các ngưỡng này là phiên bản ban đầu và sẽ được đánh giá lại dựa trên tỷ lệ dropout thực tế.

In [16]:
def map_risk_level(score):
    if score >= 6:
        return "Cao"
    elif score >= 3:
        return "Trung bình"
    else:
        return "Thấp"

In [17]:
def calculate_risk(row):
    score = 0
    reasons = []
    recommendations = []

    # 1. GPA
    if row["GPA"] < 2.0:
        score += 3
        reasons.append("GPA dưới 2.0")
        recommendations.append(
            "Tư vấn học tập và lập kế hoạch cải thiện GPA"
        )
    elif row["GPA"] < 2.5:
        score += 1
        reasons.append("GPA từ 2.0 đến dưới 2.5")
        recommendations.append(
            "Theo dõi kết quả học tập trong học kỳ tiếp theo"
        )

    # 2. Tỷ lệ chuyên cần
    if row["Attendance_Rate"] < 75:
        score += 2
        reasons.append("Tỷ lệ chuyên cần dưới 75%")
        recommendations.append(
            "Liên hệ sinh viên và xác định nguyên nhân nghỉ học"
        )
    elif row["Attendance_Rate"] < 85:
        score += 1
        reasons.append("Tỷ lệ chuyên cần từ 75% đến dưới 85%")
        recommendations.append(
            "Theo dõi chuyên cần và nhắc nhở đi học đầy đủ"
        )

    # 3. Mức độ căng thẳng
    if row["Stress_Index"] >= 7:
        score += 2
        reasons.append("Mức độ căng thẳng cao")
        recommendations.append(
            "Đề xuất tư vấn tâm lý và theo dõi sức khỏe tinh thần"
        )
    elif row["Stress_Index"] >= 5:
        score += 1
        reasons.append("Mức độ căng thẳng cần theo dõi")
        recommendations.append(
            "Theo dõi mức độ căng thẳng định kỳ"
        )

    # 4. Thời gian tự học
    if row["Study_Hours_per_Day"] < 2:
        score += 1
        reasons.append("Thời gian tự học dưới 2 giờ mỗi ngày")
        recommendations.append(
            "Hỗ trợ xây dựng thời gian biểu học tập"
        )

    # 5. Nộp bài trễ
    if row["Assignment_Delay_Days"] >= 3:
        score += 1
        reasons.append("Nộp bài trễ từ 3 ngày trở lên")
        recommendations.append(
            "Nhắc hạn bài và hỗ trợ kỹ năng quản lý thời gian"
        )

    # 6. Truy cập Internet
    if row["Internet_Access"] == "No":
        score += 1
        reasons.append("Không có điều kiện truy cập Internet")
        recommendations.append(
            "Hỗ trợ thiết bị hoặc địa điểm truy cập Internet"
        )

    # 7. Công việc làm thêm
    if row["Part_Time_Job"] == "Yes":
        score += 1
        reasons.append("Công việc làm thêm có thể ảnh hưởng việc học")
        recommendations.append(
            "Tư vấn cân bằng thời gian làm thêm và học tập"
        )

    if not reasons:
        reasons.append("Không phát hiện yếu tố rủi ro đáng kể")

    if not recommendations:
        recommendations.append("Tiếp tục theo dõi định kỳ")

    return pd.Series({
        "Risk_Score": score,
        "Risk_Level": map_risk_level(score),
        "Risk_Reasons": "; ".join(reasons),
        "Recommendations": "; ".join(recommendations)
    })

In [18]:
sample_student = df_clean.iloc[1]
sample_result = calculate_risk(sample_student)

print("Student ID:", sample_student["Student_ID"])
print("Dropout thực tế:", sample_student["Dropout"])

display(sample_result.to_frame(name="Kết quả"))

Student ID: 2
Dropout thực tế: 1


,Kết quả
Risk_Score,6
Risk_Level,Cao
Risk_Reasons,GPA dưới 2.0; Tỷ lệ chuyên cần dưới 75%; Mức đ...
Recommendations,Tư vấn học tập và lập kế hoạch cải thiện GPA; ...


### 10. Áp dụng Rule-based Scoring cho toàn bộ dữ liệu

Hàm chấm điểm được áp dụng cho từng sinh viên. Kết quả được ghép với dữ liệu ban đầu để tạo các cột điểm rủi ro, mức cảnh báo, nguyên nhân và khuyến nghị.

In [19]:
risk_results = df_clean.apply(
    calculate_risk,
    axis=1
)

df_scored = pd.concat(
    [df_clean, risk_results],
    axis=1
)

print("Kích thước dữ liệu sau khi chấm điểm:", df_scored.shape)

df_scored[
    [
        "Student_ID",
        "Dropout",
        "Risk_Score",
        "Risk_Level",
        "Risk_Reasons",
        "Recommendations"
    ]
].head()

Kích thước dữ liệu sau khi chấm điểm: (10000, 23)


,Student_ID,Dropout,Risk_Score,Risk_Level,Risk_Reasons,Recommendations
0,1,0,5,Trung bình,GPA dưới 2.0; Mức độ căng thẳng cần theo dõi; ...,Tư vấn học tập và lập kế hoạch cải thiện GPA; ...
1,2,1,6,Cao,GPA dưới 2.0; Tỷ lệ chuyên cần dưới 75%; Mức đ...,Tư vấn học tập và lập kế hoạch cải thiện GPA; ...
2,3,0,7,Cao,GPA dưới 2.0; Tỷ lệ chuyên cần dưới 75%; Mức đ...,Tư vấn học tập và lập kế hoạch cải thiện GPA; ...
3,4,1,5,Trung bình,GPA dưới 2.0; Tỷ lệ chuyên cần từ 75% đến dưới...,Tư vấn học tập và lập kế hoạch cải thiện GPA; ...
4,5,0,6,Cao,GPA dưới 2.0; Tỷ lệ chuyên cần từ 75% đến dưới...,Tư vấn học tập và lập kế hoạch cải thiện GPA; ...


### 11. Đánh giá mức độ phân tầng rủi ro

Một bộ luật hợp lý cần có tỷ lệ dropout tăng dần từ nhóm nguy cơ thấp, đến trung bình và cao.

In [20]:
risk_order = [
    "Thấp",
    "Trung bình",
    "Cao"
]

risk_evaluation = (
    df_scored
    .groupby("Risk_Level")["Dropout"]
    .agg(
        Student_Count="count",
        Dropout_Count="sum",
        Dropout_Rate="mean"
    )
    .reindex(risk_order)
)

risk_evaluation["Dropout_Rate"] = (
    risk_evaluation["Dropout_Rate"] * 100
).round(2)

risk_evaluation

,Student_Count,Dropout_Count,Dropout_Rate
Risk_Level,,,
Thấp,3127,176,5.63
Trung bình,4295,874,20.35
Cao,2578,1304,50.58


### 12. Đánh giá chiến lược cảnh báo

Hai chiến lược được so sánh:

1. Chỉ cảnh báo sinh viên nguy cơ cao.
2. Cảnh báo cả sinh viên nguy cơ trung bình và cao.

Precision thể hiện mức độ chính xác của cảnh báo. Recall thể hiện khả năng phát hiện sinh viên thực sự bỏ học.

In [21]:
def evaluate_alert(y_true, y_pred):
    true_positive = ((y_true == 1) & (y_pred == 1)).sum()
    false_positive = ((y_true == 0) & (y_pred == 1)).sum()
    false_negative = ((y_true == 1) & (y_pred == 0)).sum()
    true_negative = ((y_true == 0) & (y_pred == 0)).sum()

    precision = true_positive / (true_positive + false_positive)
    recall = true_positive / (true_positive + false_negative)
    f1_score = 2 * precision * recall / (precision + recall)
    accuracy = (true_positive + true_negative) / len(y_true)

    return pd.Series({
        "TP": true_positive,
        "FP": false_positive,
        "FN": false_negative,
        "TN": true_negative,
        "Precision (%)": precision * 100,
        "Recall (%)": recall * 100,
        "F1-score (%)": f1_score * 100,
        "Accuracy (%)": accuracy * 100
    })

In [22]:
high_only_prediction = (
    df_scored["Risk_Level"] == "Cao"
).astype(int)

medium_or_high_prediction = (
    df_scored["Risk_Score"] >= 3
).astype(int)

alert_evaluation = pd.DataFrame({
    "Chỉ cảnh báo mức Cao": evaluate_alert(
        df_scored["Dropout"],
        high_only_prediction
    ),
    "Cảnh báo Trung bình và Cao": evaluate_alert(
        df_scored["Dropout"],
        medium_or_high_prediction
    )
}).T.round(2)

alert_evaluation

,TP,FP,FN,TN,Precision (%),Recall (%),F1-score (%),Accuracy (%)
Chỉ cảnh báo mức Cao,1304.0,1274.0,1050.0,6372.0,50.58,55.40,52.88,76.76
Cảnh báo Trung bình và Cao,2178.0,4695.0,176.0,2951.0,31.69,92.52,47.21,51.29


#### Kết luận đánh giá

Bộ luật phân chia sinh viên thành ba nhóm có tỷ lệ dropout tăng rõ rệt từ 5.63% ở nhóm nguy cơ thấp, 20.35% ở nhóm trung bình và 50.58% ở nhóm cao.

Nếu chỉ xem mức Cao là cảnh báo, hệ thống đạt precision 50.58% và recall 55.40%. Nếu cảnh báo cả mức Trung bình và Cao, recall tăng lên 92.52%, phù hợp hơn với mục tiêu cảnh báo sớm nhưng tạo ra nhiều cảnh báo nhầm hơn.

Do đó:

- Mức `Cao`: ưu tiên liên hệ và can thiệp ngay.
- Mức `Trung bình`: đưa vào danh sách theo dõi.
- Mức `Thấp`: tiếp tục theo dõi định kỳ.

In [23]:
sample_outputs = pd.concat([
    df_scored[df_scored["Risk_Level"] == "Thấp"].head(1),
    df_scored[df_scored["Risk_Level"] == "Trung bình"].head(1),
    df_scored[df_scored["Risk_Level"] == "Cao"].head(1)
])

output_columns = [
    "Student_ID",
    "Dropout",
    "GPA",
    "Attendance_Rate",
    "Stress_Index",
    "Risk_Score",
    "Risk_Level",
    "Risk_Reasons",
    "Recommendations"
]

sample_outputs[output_columns]

,Student_ID,Dropout,GPA,Attendance_Rate,Stress_Index,Risk_Score,Risk_Level,Risk_Reasons,Recommendations
5,6,0,2.52,89.1,6.0,1,Thấp,Mức độ căng thẳng cần theo dõi,Theo dõi mức độ căng thẳng định kỳ
0,1,0,0.96,86.1,5.5,5,Trung bình,GPA dưới 2.0; Mức độ căng thẳng cần theo dõi; ...,Tư vấn học tập và lập kế hoạch cải thiện GPA; ...
1,2,1,1.28,68.0,6.8,6,Cao,GPA dưới 2.0; Tỷ lệ chuyên cần dưới 75%; Mức đ...,Tư vấn học tập và lập kế hoạch cải thiện GPA; ...


### 13. Xuất kết quả

Kết quả được xuất thành các file CSV để sử dụng trong báo cáo, backend và giao diện web.

In [24]:
from pathlib import Path

output_directory = Path("../outputs")
output_directory.mkdir(exist_ok=True)

rules_table.to_csv(
    output_directory / "risk_rules_table.csv",
    index=False,
    encoding="utf-8-sig"
)

df_scored[[
    "Student_ID",
    "Dropout",
    "Risk_Score",
    "Risk_Level",
    "Risk_Reasons",
    "Recommendations"
]].to_csv(
    output_directory / "student_risk_scored.csv",
    index=False,
    encoding="utf-8-sig"
)

sample_outputs[output_columns].to_csv(
    output_directory / "sample_decision_outputs.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã xuất kết quả vào thư mục outputs.")

Đã xuất kết quả vào thư mục outputs.
